# ParkShare - Challenge 48h - Partie Data

**Objectif** : Identifier les zones géographiques à fort potentiel commercial pour ParkShare (partage de places de stationnement en copropriété).

## Pipeline
1. Collecte des données ouvertes (INSEE, DVF, RNIC, BNLS, IGN)
2. Nettoyage et uniformisation sur le code commune INSEE
3. Jointure et calcul d'indicateurs dérivés
4. Scoring et production de 4 KPIs
5. Visualisations

## Sources de données
| Dataset | Source | Période |
|---------|--------|---------|
| Motorisation par commune | INSEE / data.gouv.fr | 2022 |
| Communes France (population, densité) | data.gouv.fr | 2025 |
| Logements (collectif/individuel, garages) | INSEE RP | 2022 |
| Revenus fiscaux (Filosofi) | data.gouv.fr | 2021 |
| Registre National des Copropriétés (RNIC) | ANAH / data.gouv.fr | T4 2025 |
| Demandes de Valeurs Foncières (DVF) | DGFiP / data.gouv.fr | 2025 S1 |
| Contours géographiques communes | IGN / Etalab | 2025 |
| Base Nationale Lieux de Stationnement | transport.data.gouv.fr | 2024 |

---
## Étape 1 : Collecte des données

Les données sont téléchargées depuis les APIs publiques dans `data/raw/`.

In [ ]:
import os
import subprocess

RAW = "raw"
os.makedirs(RAW, exist_ok=True)

datasets = {
    "motorisation_commune.csv": "https://static.data.gouv.fr/resources/part-des-menages-disposant-au-moins-dune-voiture-taux-de-motorisation/20260203-155004/part-des-menages-disposant-au-moins-dune-voiture-taux-de-motorisation-commune.csv",
    "communes_france.csv.gz": "https://www.data.gouv.fr/fr/datasets/r/6989ed1a-8ffb-4ef9-b008-340327c99430",
    "revenus_commune.csv": "https://static.data.gouv.fr/resources/revenu-des-francais-a-la-commune/20251210-134014/revenu-des-francais-a-la-commune-1765372688826.csv",
    "rnic_t4_2025.csv": "https://static.data.gouv.fr/resources/registre-national-dimmatriculation-des-coproprietes/20260105-114009/rnc-data-gouv-with-qpv.csv",
    "dvf_2025_s1.txt.zip": "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234902/valeursfoncieres-2025-s1.txt.zip",
    "communes_contours_100m.geojson": "https://etalab-datasets.geo.data.gouv.fr/contours-administratifs/latest/geojson/communes-100m.geojson",
    "stationnement_bnls.csv": "https://static.data.gouv.fr/resources/base-nationale-des-lieux-de-stationnement/20240109-111856/base-nationale-des-lieux-de-stationnement-outil-de-consolidation-bnls-v2.csv",
    "logements_insee.zip": "https://www.insee.fr/fr/statistiques/fichier/8581474/base-cc-logement-2022_csv.zip",
}

for filename, url in datasets.items():
    filepath = os.path.join(RAW, filename)
    if not os.path.exists(filepath):
        print(f"Téléchargement de {filename}...")
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True)
    else:
        print(f"{filename} déjà présent")

# Décompression
import zipfile, gzip, shutil

# communes_france.csv.gz -> communes_france.csv
gz_path = os.path.join(RAW, "communes_france.csv.gz")
csv_path = os.path.join(RAW, "communes_france.csv")
if os.path.exists(gz_path) and not os.path.exists(csv_path):
    with gzip.open(gz_path, 'rb') as f_in, open(csv_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print("communes_france.csv décompressé")

# DVF zip
dvf_zip = os.path.join(RAW, "dvf_2025_s1.txt.zip")
if os.path.exists(dvf_zip):
    with zipfile.ZipFile(dvf_zip, 'r') as z:
        z.extractall(RAW)
    print("DVF décompressé")

# Logements zip
log_zip = os.path.join(RAW, "logements_insee.zip")
if os.path.exists(log_zip):
    with zipfile.ZipFile(log_zip, 'r') as z:
        z.extractall(os.path.join(RAW, "logements_insee"))
    print("Logements INSEE décompressé")

print("\nFichiers dans raw/:")
for f in sorted(os.listdir(RAW)):
    if os.path.isfile(os.path.join(RAW, f)):
        size = os.path.getsize(os.path.join(RAW, f)) / 1024 / 1024
        print(f"  {f:45s} {size:6.1f} MB")

---
## Étape 2 : Nettoyage des données

Chaque dataset est nettoyé, les colonnes utiles extraites, et le code commune uniformisé sur 5 caractères.

In [ ]:
# Exécution du script de nettoyage
%run clean_datasets.py

In [ ]:
import pandas as pd

# Aperçu des fichiers nettoyés
CLEANED = "cleaned"
for f in sorted(os.listdir(CLEANED)):
    if f.endswith('.csv'):
        df_tmp = pd.read_csv(os.path.join(CLEANED, f), nrows=3)
        print(f"\n{'='*60}")
        print(f"{f} ({len(df_tmp.columns)} colonnes)")
        print(f"{'='*60}")
        display(df_tmp)

---
## Étape 3 & 4 : Jointure, indicateurs dérivés et KPIs

Fusion de tous les datasets sur le code commune INSEE, calcul des indicateurs dérivés et du scoring.

### Méthode de scoring

Le **score de potentiel** est un composite pondéré de 7 facteurs normalisés (min-max 0-100) :

| Facteur | Poids | Justification |
|---------|-------|---------------|
| Densité de copropriétés (copro/km²) | 20% | Concentration des cibles commerciales |
| Taux de ménages sans garage | 20% | Demande latente de stationnement |
| Part logements collectifs | 15% | Parc adapté au modèle ParkShare |
| Lots parking / 1000 hab | 15% | Offre de parking existante en copro, partageable |
| Taux de motorisation | 10% | Besoin de stationnement |
| Densité de population | 10% | Tension urbaine |
| Part copro avec parking | 10% | Infrastructure parking existante |

### 4 KPIs produits

1. **Score de potentiel** — score composite 0-100 par commune
2. **Classement Top N** — communes prioritaires pour la prospection
3. **Indice de tension stationnement** — motorisation × sans_garage × densité
4. **Densité d'opportunité** — copropriétés par km²

In [ ]:
# Exécution de la jointure et du scoring
%run build_kpis.py

In [ ]:
# Chargement du résultat
OUTPUT = "output"
df_kpis = pd.read_csv(os.path.join(OUTPUT, "kpis_par_commune.csv"), dtype={"code_insee": str})

print(f"Table finale : {len(df_kpis)} communes, {len(df_kpis.columns)} colonnes")
print(f"\nStatistiques du score de potentiel :")
display(df_kpis["score_potentiel"].describe())

print(f"\nTop 20 communes :")
display(df_kpis.head(20)[[
    "rang", "nom_standard", "dep_nom", "population",
    "nb_coproprietes", "total_lots_stationnement",
    "taux_motorisation", "part_collectif", "taux_voiture_sans_garage",
    "score_potentiel"
]])

---
## Étape 5 : Visualisations

In [ ]:
# Exécution des visualisations
%run build_visualizations.py

In [ ]:
from IPython.display import Image, display, HTML

print("Top 20 communes par score de potentiel")
display(Image(filename="output/top20_score.png", width=800))

In [ ]:
print("Scatter : Motorisation vs Densité de copropriétés")
display(Image(filename="output/scatter_motorisation_copro.png", width=800))

In [ ]:
print("Heatmap de corrélation")
display(Image(filename="output/heatmap_correlation.png", width=800))

In [ ]:
print("Distribution du score")
display(Image(filename="output/distribution_score.png", width=800))

In [ ]:
print("Top 15 départements")
display(Image(filename="output/top15_departements.png", width=800))

In [ ]:
# Carte interactive (ouvrir carte_score.html dans un navigateur)
print("Carte interactive disponible : output/carte_score.html")
print("Ouvrir dans un navigateur pour naviguer sur la carte.")

---
## Analyse des résultats

### Zones prioritaires identifiées

Les zones à fort potentiel pour ParkShare se concentrent sur :

1. **Petite couronne parisienne** (Hauts-de-Seine, Val-de-Marne, Seine-Saint-Denis) : forte densité de copropriétés, faible taux de garage, tension stationnement élevée

2. **Agglomérations Haute-Savoie / frontalières** (Annecy, Annemasse, Évian, Ambilly) : forte motorisation + copropriétés récentes avec parking

3. **Côte d'Azur** (Antibes, Le Cannet, Beaulieu-sur-Mer, Saint-Laurent-du-Var) : densité de copropriétés très élevée, prix du stationnement élevé

4. **Agglomération strasbourgeoise** (Bischheim, Schiltigheim) : copropriétés denses avec bon taux de motorisation

### Corrélations clés

- Le score corrèle le plus avec la **part de logements collectifs** (0.86) et la **densité de copropriétés** (0.51)
- La **motorisation** est anti-corrélée (-0.31) : les zones très urbaines sont moins motorisées mais ont plus de demande
- Le **taux de ménages sans garage** est un excellent prédicteur de la demande latente
- **Part avec garage** et **sans garage** sont très anti-corrélés (-0.92) : complémentaires

---
## Fichiers livrés

| Fichier | Description |
|---------|-------------|
| `data/raw/` | Données brutes téléchargées (traçabilité) |
| `data/cleaned/` | 7 CSV nettoyés et agrégés par commune |
| `data/output/kpis_par_commune.csv` | Table finale : 13 024 communes, 93 colonnes, avec scores |
| `data/output/top30_zones.csv` | Top 30 zones prioritaires |
| `data/output/communes_scored.geojson` | GeoJSON enrichi pour la carte du dashboard (Dev) |
| `data/output/carte_score.html` | Carte interactive Plotly |
| `data/output/*.png` | 5 graphiques d'analyse |
| `data/notebook.ipynb` | Ce notebook (reproductible) |
| `data/requirements.txt` | Dépendances Python |
| `data/README.md` | Documentation complète |